In [ ]:
import json
from typing import Any

from copilotkit import CopilotKitMiddleware
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool as lc_tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver


# ── Data-fetching tool (placeholder for a real database/API call) ────
@tool
def get_sales_data() -> str:
    """Fetch current sales metrics and revenue data.

    Returns sales data including revenue, customers, conversion rates,
    and breakdowns by category and month.
    """
    # Placeholder: in production, this would query your actual database or API.
    return json.dumps({
        "totalRevenue": "$1.2M",
        "newCustomers": 3842,
        "conversionRate": "3.6%",
        "revenueByCategory": [
            {"label": "Electronics", "value": 420000},
            {"label": "Clothing", "value": 310000},
            {"label": "Home & Garden", "value": 185000},
            {"label": "Sports", "value": 160000},
            {"label": "Books", "value": 125000},
        ],
        "monthlySales": [
            {"label": "Jan", "value": 85000},
            {"label": "Feb", "value": 92000},
            {"label": "Mar", "value": 108000},
            {"label": "Apr", "value": 95000},
            {"label": "May", "value": 115000},
            {"label": "Jun", "value": 125000},
        ],
    })

In [ ]:
graph = create_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[get_sales_data],
    middleware=[CopilotKitMiddleware()],
    checkpointer=MemorySaver(),
    system_prompt=(
        "You are a helpful assistant that creates rich visual UI.\n\n"
        "Tool guidance:\n"
        "- For sales/business data requests: first call get_sales_data to fetch "
        "the latest metrics, then call generate_a2ui to visualize the results "
        "as a dashboard with charts, metrics, and cards.\n"
        "- For other rich UI: call generate_a2ui directly.\n\n"
        "IMPORTANT: After calling a tool, do NOT repeat or summarize the data "
        "in your text response. The tool renders UI automatically. "
        "Just confirm what was rendered."
    ),
)

agent.graph = graph
print("✓ Agent graph updated!")

In [ ]:
%%writefile frontend/server.ts

import { serve } from "@hono/node-server";
import {
  CopilotRuntime,
  createCopilotEndpoint,
} from "@copilotkit/runtime/v2";
import { LangGraphHttpAgent } from "@copilotkit/runtime/langgraph";

const langGraphAgent = new LangGraphHttpAgent({ url: "http://localhost:8004" });

const runtime = new CopilotRuntime({
  agents: { default: langGraphAgent },
  a2ui: { injectA2UITool: true },
});

const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

serve({ fetch: app.fetch, port: 4004 }, () => {
  console.log("\u2713 CopilotKit API server running at http://localhost:4004");
});